# Batch Suite2p with high-resolution still Cellpose masks

Pass one dataset root and shared parameters. Each result is written beside its video under `suite2p_still/plane0`, leaving existing `suite2p/plane0` outputs untouched. Run the preflight first.

In [ ]:
from collections import Counter
from pathlib import Path

from suite2p.still_cellpose_batch import (
    batch_still_cellpose,
    discover_experiments,
)

In [ ]:
ROOT = Path(r"C:\Users\mzinn1\Desktop\GCaMP6s_EX37x_Days_Repeating")

# General Suite2p parameters. Add any nested settings you want to override.
SUITE2P_SETTINGS = {
    "fs": 15.0,
    "tau": 1.0,
    "torch_device": "cuda",
    "diameter": [12.0, 12.0],
    "registration": {
        "nonrigid": True,
    },
    "extraction": {
        "neuropil_coefficient": 0.7,
    },
}

# Acquisition/database parameters.
DB_PARAMETERS = {
    "nplanes": 1,
    "nchannels": 1,
    "functional_chan": 1,
}

# Still-image Cellpose and alignment parameters.
CELLPOSE_MODEL = "cpdino"
STILL_DIAMETER = None
CELLPROB_THRESHOLD = 0.0
FLOW_THRESHOLD = 0.4
STILL_CHANNEL = 1       # zero-based second channel
STILL_CHANNEL_AXIS = 0  # used only for TIFF stills
# Required only when an OIR has a non-singleton T, L, or Z axis.
OIR_AXIS_INDICES = {}  # e.g. {"Z": 0, "T": 0}
OIR_ENVIRONMENT = "image_conversion"
DY, DX = 1, -1

# Known bad acquisitions to omit from both preflight and processing.
EXCLUDE_EXPERIMENTS = {"1-4_Day2"}


## Discover and preflight
OIR snapshots are read directly with `oirfile`. The selected 2-D channel is cached beside each OIR as `<video stem>_snap.tif`, so later runs can reuse the conversion. OIR axes are selected by their names rather than inferred from shape.

In [ ]:
experiments = discover_experiments(ROOT)
print(f"Discovered {len(experiments)} videos")
for experiment in experiments:
    still = experiment.still_tiff or experiment.still_oir
    kind = "TIFF ready" if experiment.still_tiff else "OIR ready" if experiment.still_oir else "missing still"
    print(f"[{kind:21}] {experiment.name}: {still}")

In [ ]:
preflight = batch_still_cellpose(
    ROOT,
    suite2p_settings=SUITE2P_SETTINGS,
    db_parameters=DB_PARAMETERS,
    model_name_or_path=CELLPOSE_MODEL,
    diameter=STILL_DIAMETER,
    cellprob_threshold=CELLPROB_THRESHOLD,
    flow_threshold=FLOW_THRESHOLD,
    still_channel=STILL_CHANNEL,
    channel_axis=STILL_CHANNEL_AXIS,
    oir_axis_indices=OIR_AXIS_INDICES,
    oir_environment=OIR_ENVIRONMENT,
    dy=DY, dx=DX,
    exclude_names=EXCLUDE_EXPERIMENTS,
    dry_run=True,
)
print(Counter(row["status"] for row in preflight))

## Run batch
This is resumable: completed folders are skipped. Failures are recorded and processing continues. An alignment PNG and both high- and video-resolution masks are saved per experiment.

In [ ]:
results = batch_still_cellpose(
    ROOT,
    suite2p_settings=SUITE2P_SETTINGS,
    db_parameters=DB_PARAMETERS,
    model_name_or_path=CELLPOSE_MODEL,
    diameter=STILL_DIAMETER,
    cellprob_threshold=CELLPROB_THRESHOLD,
    flow_threshold=FLOW_THRESHOLD,
    still_channel=STILL_CHANNEL,
    channel_axis=STILL_CHANNEL_AXIS,
    oir_axis_indices=OIR_AXIS_INDICES,
    oir_environment=OIR_ENVIRONMENT,
    dy=DY, dx=DX,
    output_folder="suite2p_still",
    exclude_names=EXCLUDE_EXPERIMENTS,
    skip_completed=True,
    save_qc=True,
    delete_binary_after=True,
    stop_on_error=False,
)
print(Counter(row["status"] for row in results))
for row in results:
    if row["status"] in {"failed", "skipped_missing_still", "blocked_oirfile_runtime"}:
        print(row["status"], row["name"], row.get("error", ""))